In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3s81mugm/unsloth_6f1b2e227d554db3be3420f1c85c24c6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3s81mugm/unsloth_6f1b2e227d554db3be3420f1c85c24c6
  Resolved https://github.com/unslothai/unsloth.git to commit 746ad086d80e6bd4d24346d67c6d96a2f1556a64
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.7 MB/s eta 0:00:00
   ━━━

In [ ]:
import json
from collections import defaultdict
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# ==========================================
# 1. Data Preprocessing & Structuring
# ==========================================

def prepare_dataset(jsonl_file="memory_logs.jsonl"):
    """
    Reads the raw sequential memory logs and groups them into logical
    Lifecycle Blocks (entire entity trees connected to a root parent).
    """
    entity_to_root = {}
    root_to_events = defaultdict(list)
    root_to_leak_status = defaultdict(bool)
    root_to_leaked_entities = defaultdict(set)

    # 1a. Parse and group events
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            event = json.loads(line)

            ent_id = event["entity_id"]
            parent_id = event["parent_id"]

            # Resolve the root entity for grouping
            if parent_id is None:
                root_id = ent_id
                entity_to_root[ent_id] = root_id
            else:
                # Since events are generated chronologically, the parent's root is already known
                root_id = entity_to_root.get(parent_id, parent_id)
                entity_to_root[ent_id] = root_id

            root_to_events[root_id].append(event)

            # Identify if this specific entity leaked
            if event.get("is_leak", False):
                root_to_leak_status[root_id] = True
                root_to_leaked_entities[root_id].add(ent_id)

    # 1b. Format into Instruction/Input/Output pairs
    instructions = []

    for root_id, events in root_to_events.items():
        # Ensure chronological order for the model
        events.sort(key=lambda x: x["tick"])

        prompt_events = []
        for e in events:
            # We MUST strip the 'is_leak' label from the prompt so the LLM
            # actually learns to detect the missing FREE_MEMORY event logically.
            safe_event = e.copy()
            safe_event.pop("is_leak", None)
            prompt_events.append(safe_event)

        input_text = json.dumps(prompt_events, indent=2)

        if root_to_leak_status[root_id]:
            orphans = ", ".join(sorted(list(root_to_leaked_entities[root_id])))
            output_text = f"Memory leak detected! Orphaned entities: {orphans}. Missing FREE_MEMORY events."
        else:
            output_text = "No leak detected. Memory cleanup successful."

        instructions.append({
            "instruction": "Analyze the following memory lifecycle events for this entity tree. Determine if there is a memory leak and identify any orphaned entities.",
            "input": input_text,
            "output": output_text
        })

    return Dataset.from_list(instructions)

def main():
    # ==========================================
    # 2. Unsloth Model Setup
    # ==========================================
    max_seq_length = 2048 # Optimized for standard context windows

    # Load model and tokenizer with 4-bit quantization via QLoRA
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/llama-3-8b-bnb-4bit",
        max_seq_length = max_seq_length,
        dtype = None, # Auto-detects fp16 or bf16 based on hardware
        load_in_4bit = True,
    )

    # Alpaca-style prompt template for Instruction Tuning
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

    def format_prompts(examples):
        instructions = examples["instruction"]
        inputs       = examples["input"]
        outputs      = examples["output"]
        texts = []
        for instruction, input, output in zip(instructions, inputs, outputs):
            # Append the EOS token so the model learns exactly when to stop generating
            text = alpaca_prompt.format(instruction, input, output) + tokenizer.eos_token
            texts.append(text)
        return { "text" : texts }

    print("Preparing dataset...")
    raw_dataset = prepare_dataset("memory_logs.jsonl")
    dataset = raw_dataset.map(format_prompts, batched=True)

    # ==========================================
    # 3. LoRA Configuration
    # ==========================================
    # Apply PEFT adapters to target attention and MLP matrices
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth", # Enables long context with less VRAM
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )

    # ==========================================
    # 4. Training Pipeline
    # ==========================================
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False, # Set to False to strictly isolate examples
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            warmup_steps = 5,
            max_steps = 60, # Quick test run as requested
            learning_rate = 2e-4,
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 1,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs",
        ),
    )

    print("Starting SFT Training...")
    trainer.train()

    # ==========================================
    # 5. Export
    # ==========================================
    export_path = "lora_memory_profiler"
    print(f"Saving LoRA adapters to {export_path}...")

    # Save the fine-tuned PEFT model and tokenizer locally
    model.save_pretrained(export_path)
    tokenizer.save_pretrained(export_path)

    print("Done! Model successfully trained and exported.")

if __name__ == "__main__":
    main()

==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Preparing dataset...


Map:   0%|          | 0/977 [00:00<?, ? examples/s]

...

Step,Training Loss
1,1.723281
2,1.578553
3,1.785850
4,1.719869
5,1.442177
6,1.409587
7,1.418493
8,1.365210
9,1.304888
10,1.261959


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


Saving LoRA adapters to lora_memory_profiler...


Unsloth: Restored added_tokens_decoder metadata in lora_memory_profiler/tokenizer_config.json.


Done! Model successfully trained and exported.


In [ ]:
# Unsloth ve gerekli bağımlılıkları sisteme yeniden kuralım
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-f25fmoar/unsloth_f274b567b9de4c2291b45cc43fa7fe7c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-f25fmoar/unsloth_f274b567b9de4c2291b45cc43fa7fe7c
  Resolved https://github.com/unslothai/unsloth.git to commit 746ad086d80e6bd4d24346d67c6d96a2f1556a64
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.3 MB/s eta 0:00:00
   ━━

In [ ]:
from unsloth import FastLanguageModel

# 1. Eğitilmiş LoRA adaptörünü ve ana modeli yeniden yüklüyoruz.
# Lütfen "lora_memory_profiler" kısmını, bir önceki görselde adaptörü
# kaydettiğin tam klasör adıyla eşleştiğinden emin ol.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_memory_profiler",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# 2. Modeli Ollama uyumlu GGUF formatına dönüştürüyoruz.
print("GGUF dönüştürme işlemi başlıyor, bu biraz zaman alabilir...")
model.save_pretrained_gguf("memory_profiler_model", tokenizer, quantization_method = "q4_k_m")
print("İşlem tamamlandı! Sol taraftaki klasör simgesinden .gguf dosyasını indirebilirsin.")

==((====))==  Unsloth 2026.8.19: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load lora_memory_profiler as a legacy tokenizer.


GGUF dönüştürme işlemi başlıyor, bu biraz zaman alabilir...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in memory_profiler_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [13:20<00:00, 200.09s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:50<00:00, 102.70s/it]


Unsloth: Merge process complete. Saved to `/content/memory_profiler_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10472-mix-4b653db (app-b10472-mix-4b653db-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['memory_profiler_model_gguf/llama-3-8b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions complet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!cp /content/memory_profiler_model_gguf/llama-3-8b.Q4_K_M.gguf /content/drive/MyDrive/